## Week 2 Day 2 - Orchestration

Our first Agentic Framework project!!

### Part 1: Email Setup

### Part 2: Orchestrating by code

### Part 3: Orchestrating by LLMs

- 3a: via Tools
- 3b: via Handoffs

## Part 1: Email Setup

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending Emails</h2>
            <span style="color:#ff7800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>

## Setting up to send emails from your SMTP server

### STEP 1: Determining your SMTP Server

Either google or ask ChatGPT / Claude for the SMTP server for your email. Here are some common ones. Some email providers might not have SMTP servers enabled (eg Microsoft 365 for work/school).

Google: smtp.gmail.com  
Outlook.com / Hotmail / Live: smtp-mail.outlook.com  
Microsoft 365: smtp.office365.com  
iCloud Mail: smtp.mail.me.com  

Add to your .env file:  

`EMAIL_SMTP_SERVER=xxxx`

### STEP 2: Obtain an app specific password

Google how to do this for your email provider. For gmail, you need to have 2-step verification on. Then visit this page:

https://myaccount.google.com/apppasswords

Give it any name; copy the password and add it to your .env file, removing the spaces that it adds. (It should be 16 characters, no spaces.)

`EMAIL_APP_PASSWORD=xxxx`

### STEP 3: Add in your email address:

`EMAIL_ADDRESS=xxx`

Remember to Save the .env file!

In [14]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage

load_dotenv(override=True)
ModelName = "gpt-5.4-mini"

In [15]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

In [16]:
def SendEmail(Subject: str, TextBody: str, HTMLBody: str):
    """Send an email"""
    Message = EmailMessage()
    Message["From"] = EMAIL_ADDRESS
    Message["To"] = EMAIL_ADDRESS
    Message["Subject"] = Subject
    Message.set_content(TextBody)
    Message.add_alternative(HTMLBody, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as Server:
        Server.starttls()
        Server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        Server.send_message(Message)

In [8]:
SendEmail("Progress Message", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

An excellent write-up is here:

https://openai.github.io/openai-agents-python/multi_agent/

We will start with by Code.

## Part 2: Orchestrating by Code

In [17]:
ModelName = "gpt-4o-mini"

CommonIntro = """
              You are a sales agent working for ComplAI, a company that provides a SaaS tool
              for ensuring SOC2 compliance and preparing for audits, powered by AI. You
              write cold sales emails.
              """

Instructions1 = CommonIntro + "You write in a professional, serious style with gravitas"
Instructions2 = CommonIntro + "You write witty, engaging and humorous emails likely to get a response"
Instructions3 = CommonIntro + "You write concise, to the point emails in the style of a busy executive"

SalesAgent1 = Agent(name="Professonal Sales Agent", instructions=Instructions1, model=ModelName)
SalesAgent2 = Agent(name="Witty Sales Agent", instructions=Instructions2, model=ModelName)
SalesAgent3 = Agent(name="Executive Sales Agent", instructions=Instructions3, model=ModelName)

In [ ]:
Result = Runner.run_streamed(SalesAgent1, input="Write a cold sales email")
async for event in Result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print (event.data.delta, end="", flush=True)

In [18]:
import sendgrid
from sendgrid.helpers.mail import Email, Mail, To, Content

@function_tool
def SendEmail(Subject: str, HTMLBody: str):
    """ Send an email to a prospective client """
    SG = sendgrid.SendGridAPIClient(api_key = os.getenv("SENDGRID_API_KEY"))
    FromEmail = Email("spraybubble@gmail.com")
    ToEmail = To("spraybubble@gmail.com")
    content = Content("text/html",HTMLBody)
    mail = Mail(FromEmail, ToEmail, Subject, content).get()
    SG.client.mail.send.post(request_body=mail)
    return {"Mail sent":"OK"}

In [ ]:
## Have each of the 3 agents write a mail. Then use another LLM 
## to evaluate which is the best email and use the email tool to send the mail

Message = "Write a cold sales email"

DecisionInstructions = """
                       You are a sales manager working for ComplAI, a company that provides a SaaS tool
                       for ensuring SOC2 compliance and preparing for audits, powered by AI. You have to
                       send a cold email to a sales prospect.
                       You are given drafts of 3 mails written by 3 different agents. You must decide 
                       which of these 3 mails is the most effectivem and likely to get a response from
                       the recipient
                       Please note that you have to select ONLY ONE email. Also, do not draft the email yourself.
                       You must choose one of the 3 email drafts provided to you
                       Once you have selected the mail to be sent, craft a subject line for it and convert
                       the email body into well-formatted, highly engaging HTML format.
                       Use the SendEmail tool to send the message. 
                       """

with trace("Sales Manager mail"):
    Results = await asyncio.gather(
        Runner.run(SalesAgent1, Message),
        Runner.run(SalesAgent2, Message),
        Runner.run(SalesAgent3, Message)
    )

    Outputs = [Result.final_output for Result in Results]
    print ("\n\n".join(Outputs))

    CombinedEmails = "## Set of cold email drafts: \n\n" + "\n## Email Draft \n\n".join(Outputs)

    SalesManager = Agent(name="Sales Manager", instructions=DecisionInstructions, model="gpt-5.4-mini", 
                              tools=[SendEmail], model_settings=ModelSettings(tool_choice="required"))
    Result = await Runner.run(SalesManager, CombinedEmails)
    print (Result)



Subject: Streamline Your SOC 2 Compliance Process with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

As organizations increasingly prioritize data security and compliance, navigating the complexities of SOC 2 requirements can become a daunting task. At ComplAI, we understand the challenges you face in ensuring compliance while maintaining operational efficiency.

Our cutting-edge SaaS tool leverages AI to simplify the entire SOC 2 compliance process. By automating documentation, risk assessments, and audit preparations, you can significantly reduce the time and resources spent on compliance activities. This not only enhances your operational efficiency but also empowers your team to focus on what truly matters—growing your business.

We would welcome the opportunity to discuss how ComplAI can be tailored to meet your organization’s specific needs and help you achieve compliance with confidence. 

Would you be open to a brief call next week to explore this furt

Now go and check out the trace:

https://platform.openai.com/traces

## Part 3: Orchestrating by LLMs

### 3a: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

The OpenAI Agents SDK gives a very simple way to do this.

This works best when the flow is:

Agent A -> Agent B -> Agent A

And for the classic "Planning Agent" situation.

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [ ]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

## And now it's time for our Sales Manager - our planning agent

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)


In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!! Also look in your Junk / Spam folder - after all, this is basically a spam message..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Unreliable results?</h2>
            <span style="color:#ff7800;">This is par for the course with Agentic AI, and particularly with Orchestrating by LLM.
            To resolve it, you'll need to experiment and iterate with the prompts. Particularly with smaller models, more experiments
            may be needed to get reliable outcomes.
            </span>
        </td>
    </tr>
</table>

## Part 3: Orchestrating by LLMs

### 3a: via Handoffs

I am not a fan of handoffs. They seem very unreliable. They're not used consistently by other frameworks.

Behind the scenes, OpenAI Agents SDK has implemented these with Tools anyway.

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

A -> B -> A

With handoffs, control passes across

A -> B

In [ ]:

instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)


In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

Note that handoffs can be unrealiable and a little bit frustrating. I needed to force the tool use otherwise this didn't work. If you don't get reliable behavior, try iterating on the prompts - or use a larger model. And enjoy the process; this is what Agentic AI is all about!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Try extending with some more orchestration - perhaps another iteration of the email to refine it. 
            Use Orchestrating via Code and Orchestrating via LLMs.  
            HARD CHALLENGE: switch to using a professional email provider like SendGrid or Resend. Then handle the user replying. 
            Then have the SDR respond to keep the conversation going!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>